# 01. 통계분석/머신러닝 준비

## 프로젝트 주제
**문화시설 활성화에 대한 운영·이용 현황 성과 분석**

## 이 노트북의 목적
EDA 결과를 바탕으로 통계분석과 머신러닝에 들어가기 전에 다음 내용을 정리한다.

1. 목적 변수 설정
2. 입력 후보 변수 정리
3. 제외 변수 정리
4. 결측 처리 방향 확인
5. 모델링용 데이터셋 A/B/C 구성

> 이 노트북은 EDA를 추가로 진행하는 파일이 아니라, **모델링 전 변수 설계와 데이터셋 준비용 노트북**이다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. 데이터 불러오기

EDA에서 전처리 완료한 CSV 파일을 사용한다.

- 기본 경로: `data/processed/facility_eda_preprocessed.csv`
- 업로드 환경에서는 `/mnt/data/facility_eda_preprocessed.csv`를 보조 경로로 사용한다.

In [2]:
DATA_PATH = Path("data/processed/facility_eda_preprocessed.csv")

# ChatGPT / Colab / 임시 업로드 환경에서 실행할 경우를 위한 보조 경로
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/facility_eda_preprocessed.csv")

df = pd.read_csv(DATA_PATH)

print("data path:", DATA_PATH)
print("shape:", df.shape)
display(df.head())

data path: data\processed\facility_eda_preprocessed.csv
shape: (1191, 64)


,facility_type,ID,FCLTY_NM,CTPRVN_NM,SIGNGU_NM,LEGALDONG_NM,ADSTRD_NM,FCLTY_LO,FCLTY_LA,OPNNG_DE,OPNNG_DE_RAW,OPNNG_DE_PRECISION,OPNNG_DAY_CO,BASE_DE,LAST_CHG_DE,LND_AR_VALUE,EDC_FCLTY_AR_VALUE,DATA_SPCE_AR_VALUE,NMPR_CO,QUALF_HOLD_CO,GNRL_GVRNM_EMP_CO,CNTRCT_EMP_CO,PUBLIC_VLNTER_CO,ARTGR_EMP_CO,PRFSN_CO,PRVATE_VLNTER_CO,DATA_CO,TOT_PROGRM_CO,VIEWNG_NMPR_CO,DAY_AVRG_VIEWNG_NMPR_CO,MOBILE_PROVD_AT,SOUND_PROVD_AT,HMPG_ADDR,TEL_NO,OPNNG_DAY_CO_IQR_OUTLIER,LND_AR_VALUE_IQR_OUTLIER,EDC_FCLTY_AR_VALUE_IQR_OUTLIER,DATA_SPCE_AR_VALUE_IQR_OUTLIER,NMPR_CO_IQR_OUTLIER,QUALF_HOLD_CO_IQR_OUTLIER,GNRL_GVRNM_EMP_CO_IQR_OUTLIER,CNTRCT_EMP_CO_IQR_OUTLIER,PUBLIC_VLNTER_CO_IQR_OUTLIER,ARTGR_EMP_CO_IQR_OUTLIER,PRFSN_CO_IQR_OUTLIER,PRVATE_VLNTER_CO_IQR_OUTLIER,DATA_CO_IQR_OUTLIER,TOT_PROGRM_CO_IQR_OUTLIER,VIEWNG_NMPR_CO_IQR_OUTLIER,DAY_AVRG_VIEWNG_NMPR_CO_IQR_OUTLIER,IQR_OUTLIER_FLAG_COUNT,OPNNG_YEAR,BASE_YEAR,OPERATING_YEARS,REGION_GROUP,SIDO_SIGUNGU,LOG1P_LND_AR_VALUE,LOG1P_EDC_FCLTY_AR_VALUE,LOG1P_DATA_SPCE_AR_VALUE,LOG1P_NMPR_CO,LOG1P_DATA_CO,LOG1P_TOT_PROGRM_CO,LOG1P_VIEWNG_NMPR_CO,LOG1P_DAY_AVRG_VIEWNG_NMPR_CO
0,박물관,KCDMMUS23N000000001,국립중앙박물관,서울,용산구,용산동6가,한강로동,126.977740,37.524702,1945-12-03,1945.12.03,day,360.0,2025-03-27,2025-03-27,285991.00,4975.00,1552.0,89.0,5.0,128.0,NaN,213.0,NaN,NaN,NaN,157323.0,16.0,4180285.0,11612.0,O,O,www.museum.go.kr,02-2077-9000,False,True,True,True,True,True,True,NaN,True,NaN,NaN,NaN,True,False,True,True,10,1945,2025,80,수도권,서울 용산구,12.563719,8.512382,7.347944,4.499810,11.966063,2.833213,15.245890,9.359880
1,박물관,KCDMMUS23N000000002,국립민속박물관,서울,종로구,세종로,청운효자동,126.978890,37.581675,1946-04-25,1946.04.25,day,362.0,2025-03-27,2025-03-27,39626.62,1376.68,324.0,58.0,1.0,37.0,8.0,81.0,NaN,NaN,NaN,92786.0,70.0,1307690.0,3612.0,O,X,www.nfm.go.kr,02-3704-3114,False,False,True,True,True,False,True,False,False,NaN,NaN,NaN,True,True,True,True,8,1946,2025,79,수도권,서울 종로구,10.587282,7.228156,5.783825,4.077537,11.438062,4.262680,14.083774,8.192294
2,박물관,KCDMMUS23N000000003,국립민속박물관 파주\n(개방형 수장고 및 정보센터),경기,파주시,탄현면 법흥리,탄현면,126.693918,37.786670,2021-07-23,2021.07.23,day,313.0,2025-03-27,2025-03-27,60212.70,277.49,NaN,NaN,NaN,NaN,NaN,37.0,NaN,NaN,NaN,NaN,10.0,83306.0,266.0,X,X,www.nfm.go.kr,031-580-5800,False,True,False,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,False,False,False,1,2021,2025,4,수도권,경기 파주시,11.005655,5.629382,NaN,NaN,NaN,2.397895,11.330288,5.587249
3,박물관,KCDMMUS23N000000004,대한민국역사박물관,서울,종로구,세종로,종로1.2.3.4가동,126.978339,37.573714,2012-12-26,2012.12.26,day,362.0,2025-03-27,2025-03-27,6445.00,496.00,81.0,55.0,23.0,NaN,4.0,54.0,NaN,NaN,NaN,18065.0,34.0,788347.0,2178.0,O,O,www.much.go.kr,02-3703-9200,False,False,False,False,True,True,NaN,False,False,NaN,NaN,NaN,False,True,True,True,5,2012,2025,13,수도권,서울 종로구,8.771215,6.208590,4.406719,4.025352,9.801787,3.555348,13.577695,7.686621
4,박물관,KCDMMUS23N000000005,국립한글박물관,서울,용산구,용산동6가,서빙고동,126.980486,37.521069,2014-10-09,2014.10.09,day,362.0,2025-03-27,2025-03-27,285991.00,378.00,220.0,33.0,NaN,19.0,13.0,64.0,NaN,NaN,NaN,30059.0,33.0,397107.0,1097.0,O,O,www.hangeul.go.kr,02-2124-6200,False,True,False,True,True,NaN,True,True,False,NaN,NaN,NaN,True,True,True,True,9,2014,2025,11,수도권,서울 용산구,12.563719,5.937536,5.398163,3.526361,10.310951,3.526361,12.891964,7.001246


## 2. 기본 구조 확인

In [3]:
display(df.info())

summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_rate": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique()
}).sort_values("missing_rate", ascending=False)

display(summary)

<class 'pandas.DataFrame'>
RangeIndex: 1191 entries, 0 to 1190
Data columns (total 64 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   facility_type                        1191 non-null   str    
 1   ID                                   1191 non-null   str    
 2   FCLTY_NM                             1191 non-null   str    
 3   CTPRVN_NM                            1191 non-null   str    
 4   SIGNGU_NM                            1191 non-null   str    
 5   LEGALDONG_NM                         1191 non-null   str    
 6   ADSTRD_NM                            1191 non-null   str    
 7   FCLTY_LO                             1191 non-null   float64
 8   FCLTY_LA                             1191 non-null   float64
 9   OPNNG_DE                             1191 non-null   str    
 10  OPNNG_DE_RAW                         1191 non-null   str    
 11  OPNNG_DE_PRECISION                   1191

None

,dtype,missing_count,missing_rate,n_unique
PRVATE_VLNTER_CO,float64,1074,90.18,32
PRVATE_VLNTER_CO_IQR_OUTLIER,object,1074,90.18,2
CNTRCT_EMP_CO_IQR_OUTLIER,object,1019,85.56,2
CNTRCT_EMP_CO,float64,1019,85.56,12
PUBLIC_VLNTER_CO,float64,1008,84.63,80
PUBLIC_VLNTER_CO_IQR_OUTLIER,object,1008,84.63,2
PRFSN_CO_IQR_OUTLIER,object,983,82.54,2
PRFSN_CO,float64,983,82.54,14
GNRL_GVRNM_EMP_CO,float64,901,75.65,30
GNRL_GVRNM_EMP_CO_IQR_OUTLIER,object,901,75.65,2


## 3. 목적 변수 설정

EDA 결과, 관람객 수(`VIEWNG_NMPR_CO`)는 오른쪽 꼬리가 긴 분포를 보였으므로  
모델링에서는 로그 변환 변수인 `LOG1P_VIEWNG_NMPR_CO`를 기본 목적 변수로 사용한다.

### 목적 변수
- 머신러닝/회귀분석용: `LOG1P_VIEWNG_NMPR_CO`
- 해석용 원자료: `VIEWNG_NMPR_CO`

### 주의
`DAY_AVRG_VIEWNG_NMPR_CO`, `LOG1P_DAY_AVRG_VIEWNG_NMPR_CO`는 관람객 수 기반 파생 변수이므로  
관람객 수 예측 모델의 입력 변수로 사용하면 데이터 누수 가능성이 있다.

In [4]:
target = "LOG1P_VIEWNG_NMPR_CO"
target_original = "VIEWNG_NMPR_CO"

target_check = df[[target_original, target, "DAY_AVRG_VIEWNG_NMPR_CO", "LOG1P_DAY_AVRG_VIEWNG_NMPR_CO"]].isna().sum()
display(target_check)

print("전체 데이터 수:", len(df))
print("목적변수 결측 수:", df[target].isna().sum())
print("모델링 가능 데이터 수:", df[target].notna().sum())

VIEWNG_NMPR_CO                   54
LOG1P_VIEWNG_NMPR_CO             54
DAY_AVRG_VIEWNG_NMPR_CO          54
LOG1P_DAY_AVRG_VIEWNG_NMPR_CO    54
dtype: int64

전체 데이터 수: 1191
목적변수 결측 수: 54
모델링 가능 데이터 수: 1137


## 4. 모델링 전 제외 변수 정리

제외 변수는 크게 네 종류로 나눈다.

1. 목적 변수 및 데이터 누수 위험 변수
2. 식별자/시설명/연락처/홈페이지 등 모델 입력에 부적절한 변수
3. 날짜 원본 변수
4. 관람객 수 기반 이상치 플래그 또는 전체 이상치 카운트

In [5]:
# 1) 목적 변수 및 데이터 누수 위험 변수
leakage_exclude = [
    "VIEWNG_NMPR_CO",
    "LOG1P_VIEWNG_NMPR_CO",
    "DAY_AVRG_VIEWNG_NMPR_CO",
    "LOG1P_DAY_AVRG_VIEWNG_NMPR_CO",
    "VIEWNG_NMPR_CO_IQR_OUTLIER",
    "DAY_AVRG_VIEWNG_NMPR_CO_IQR_OUTLIER",
    "IQR_OUTLIER_FLAG_COUNT",
]

# 2) 식별자/문자 정보
id_text_exclude = [
    "ID",
    "FCLTY_NM",
    "HMPG_ADDR",
    "TEL_NO",
]

# 3) 날짜 원본/관리 변수
date_exclude = [
    "OPNNG_DE",
    "OPNNG_DE_RAW",
    "BASE_DE",
    "LAST_CHG_DE",
]

# 4) 일반 이상치 플래그
# 목적변수 관련 플래그뿐 아니라 입력변수의 이상치 플래그도 모델 입력보다는 해석 참고용으로 분리한다.
outlier_flag_cols = [col for col in df.columns if col.endswith("_IQR_OUTLIER")]

exclude_cols = sorted(set(leakage_exclude + id_text_exclude + date_exclude + outlier_flag_cols))

print("제외 변수 수:", len(exclude_cols))
display(exclude_cols)

제외 변수 수: 29


['ARTGR_EMP_CO_IQR_OUTLIER',
 'BASE_DE',
 'CNTRCT_EMP_CO_IQR_OUTLIER',
 'DATA_CO_IQR_OUTLIER',
 'DATA_SPCE_AR_VALUE_IQR_OUTLIER',
 'DAY_AVRG_VIEWNG_NMPR_CO',
 'DAY_AVRG_VIEWNG_NMPR_CO_IQR_OUTLIER',
 'EDC_FCLTY_AR_VALUE_IQR_OUTLIER',
 'FCLTY_NM',
 'GNRL_GVRNM_EMP_CO_IQR_OUTLIER',
 'HMPG_ADDR',
 'ID',
 'IQR_OUTLIER_FLAG_COUNT',
 'LAST_CHG_DE',
 'LND_AR_VALUE_IQR_OUTLIER',
 'LOG1P_DAY_AVRG_VIEWNG_NMPR_CO',
 'LOG1P_VIEWNG_NMPR_CO',
 'NMPR_CO_IQR_OUTLIER',
 'OPNNG_DAY_CO_IQR_OUTLIER',
 'OPNNG_DE',
 'OPNNG_DE_RAW',
 'PRFSN_CO_IQR_OUTLIER',
 'PRVATE_VLNTER_CO_IQR_OUTLIER',
 'PUBLIC_VLNTER_CO_IQR_OUTLIER',
 'QUALF_HOLD_CO_IQR_OUTLIER',
 'TEL_NO',
 'TOT_PROGRM_CO_IQR_OUTLIER',
 'VIEWNG_NMPR_CO',
 'VIEWNG_NMPR_CO_IQR_OUTLIER']

## 5. 입력 후보 변수 그룹 정리

처음부터 모든 변수를 넣기보다 다음 순서로 진행한다.

- **Dataset A: 기본 모델**
  - 결측률이 비교적 낮고, EDA에서 의미가 확인된 변수 중심
- **Dataset B: 확장 모델**
  - 자료공간, 소장자료 수, 인원 수 등 결측률이 중간 이상인 변수 추가
- **Dataset C: 인력 변수 포함 모델**
  - 결측률이 높은 인력 변수를 별도 실험용으로 추가

이렇게 나누면 모델 성능과 해석 가능성을 단계적으로 비교할 수 있다.

In [6]:
# Dataset A: 기본 모델용 변수
categorical_basic = [
    "facility_type",
    "CTPRVN_NM",
    # "REGION_GROUP",  # CTPRVN_NM 대신 권역 단위로 단순화하고 싶을 때 사용
    "MOBILE_PROVD_AT",
    "SOUND_PROVD_AT",
]

numeric_basic = [
    "OPERATING_YEARS",
    "OPNNG_DAY_CO",
    "LOG1P_LND_AR_VALUE",
    "LOG1P_EDC_FCLTY_AR_VALUE",
    "LOG1P_TOT_PROGRM_CO",
]

features_basic = categorical_basic + numeric_basic

# Dataset B: 규모/자료 확장 모델용 변수
numeric_extended = [
    "LOG1P_DATA_SPCE_AR_VALUE",
    "LOG1P_DATA_CO",
    "LOG1P_NMPR_CO",
]

features_extended = features_basic + numeric_extended

# Dataset C: 인력 변수 포함 실험 모델용 변수
staff_features = [
    "ARTGR_EMP_CO",
    "QUALF_HOLD_CO",
    "GNRL_GVRNM_EMP_CO",
    "CNTRCT_EMP_CO",
    "PRFSN_CO",
    "PUBLIC_VLNTER_CO",
    "PRVATE_VLNTER_CO",
]

features_staff = features_extended + staff_features

feature_sets = {
    "A_basic": features_basic,
    "B_extended": features_extended,
    "C_staff": features_staff,
}

for name, features in feature_sets.items():
    missing_features = [col for col in features if col not in df.columns]
    print(f"{name}: 변수 {len(features)}개 / 존재하지 않는 변수 {len(missing_features)}개")
    if missing_features:
        print("  missing:", missing_features)

A_basic: 변수 9개 / 존재하지 않는 변수 0개
B_extended: 변수 12개 / 존재하지 않는 변수 0개
C_staff: 변수 19개 / 존재하지 않는 변수 0개


## 6. 목적 변수 결측 제거

모델링에서는 목적 변수 결측이 있는 행은 학습할 수 없으므로 제거한다.

In [7]:
model_df = df.dropna(subset=[target]).copy()

print("원본 데이터:", df.shape)
print("목적변수 결측 제거 후:", model_df.shape)

원본 데이터: (1191, 64)
목적변수 결측 제거 후: (1137, 64)


## 7. 후보 변수별 결측률 확인

결측률을 보고 기본 모델과 확장 모델의 안정성을 비교한다.

In [8]:
def feature_missing_table(data, features):
    return pd.DataFrame({
        "column": features,
        "dtype": [str(data[col].dtype) if col in data.columns else "NOT_FOUND" for col in features],
        "missing_count": [data[col].isna().sum() if col in data.columns else np.nan for col in features],
        "missing_rate": [round(data[col].isna().mean() * 100, 2) if col in data.columns else np.nan for col in features],
        "n_unique": [data[col].nunique() if col in data.columns else np.nan for col in features],
    }).sort_values("missing_rate", ascending=False)

for name, features in feature_sets.items():
    print(f"\n[{name}]")
    display(feature_missing_table(model_df, features))


[A_basic]


,column,dtype,missing_count,missing_rate,n_unique
7,LOG1P_EDC_FCLTY_AR_VALUE,float64,303,26.65,459
8,LOG1P_TOT_PROGRM_CO,float64,242,21.28,59
6,LOG1P_LND_AR_VALUE,float64,10,0.88,1060
2,MOBILE_PROVD_AT,str,6,0.53,2
3,SOUND_PROVD_AT,str,5,0.44,3
5,OPNNG_DAY_CO,float64,1,0.09,198
0,facility_type,str,0,0.00,2
4,OPERATING_YEARS,int64,0,0.00,70
1,CTPRVN_NM,str,0,0.00,20



[B_extended]


,column,dtype,missing_count,missing_rate,n_unique
11,LOG1P_NMPR_CO,float64,504,44.33,68
10,LOG1P_DATA_CO,float64,496,43.62,398
9,LOG1P_DATA_SPCE_AR_VALUE,float64,480,42.22,261
7,LOG1P_EDC_FCLTY_AR_VALUE,float64,303,26.65,459
8,LOG1P_TOT_PROGRM_CO,float64,242,21.28,59
6,LOG1P_LND_AR_VALUE,float64,10,0.88,1060
2,MOBILE_PROVD_AT,str,6,0.53,2
3,SOUND_PROVD_AT,str,5,0.44,3
5,OPNNG_DAY_CO,float64,1,0.09,198
1,CTPRVN_NM,str,0,0.00,20



[C_staff]


,column,dtype,missing_count,missing_rate,n_unique
18,PRVATE_VLNTER_CO,float64,1023,89.97,32
15,CNTRCT_EMP_CO,float64,967,85.05,12
17,PUBLIC_VLNTER_CO,float64,954,83.91,80
16,PRFSN_CO,float64,932,81.97,14
14,GNRL_GVRNM_EMP_CO,float64,849,74.67,30
13,QUALF_HOLD_CO,float64,714,62.80,17
12,ARTGR_EMP_CO,float64,617,54.27,37
11,LOG1P_NMPR_CO,float64,504,44.33,68
10,LOG1P_DATA_CO,float64,496,43.62,398
9,LOG1P_DATA_SPCE_AR_VALUE,float64,480,42.22,261


## 8. 모델링용 데이터셋 생성

아래 코드는 변수 세트별로 `X`, `y`를 생성한다.

- `X_basic`, `y`
- `X_extended`, `y`
- `X_staff`, `y`

실제 모델 학습 시에는 train/test split 이후에 결측 대체와 인코딩을 적용해야 한다.

In [9]:
X_basic = model_df[features_basic].copy()
X_extended = model_df[features_extended].copy()
X_staff = model_df[features_staff].copy()

y = model_df[target].copy()

print("X_basic:", X_basic.shape)
print("X_extended:", X_extended.shape)
print("X_staff:", X_staff.shape)
print("y:", y.shape)

X_basic: (1137, 9)
X_extended: (1137, 12)
X_staff: (1137, 19)
y: (1137,)


## 9. Train/Test 분리

모델 성능 평가는 학습에 사용하지 않은 테스트 데이터로 확인해야 한다.

현재 데이터는 시설 유형이 박물관/미술관으로 나뉘므로, 가능하면 `facility_type` 비율을 유지하는 stratify split을 사용한다.

In [10]:
X = X_basic.copy()   # 우선 기본 모델 데이터셋으로 시작
y = model_df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=model_df["facility_type"]
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

display(X_train["facility_type"].value_counts(normalize=True).rename("train_ratio"))
display(X_test["facility_type"].value_counts(normalize=True).rename("test_ratio"))

X_train: (909, 9)
X_test: (228, 9)
y_train: (909,)
y_test: (228,)


facility_type
박물관    0.752475
미술관    0.247525
Name: train_ratio, dtype: float64

facility_type
박물관    0.754386
미술관    0.245614
Name: test_ratio, dtype: float64

## 10. 전처리 파이프라인 초안

모델링 단계에서 사용할 기본 전처리 방향은 다음과 같다.

### 수치형 변수
- 중앙값 대체
- 결측 여부 indicator 추가
- 선형모델 사용 시 표준화

### 범주형 변수
- 결측값은 `미상`으로 대체
- One-Hot Encoding

트리 기반 모델만 사용할 경우 수치형 변수 표준화는 필수는 아니지만,  
Ridge/Lasso/Linear Regression까지 비교하려면 표준화까지 포함하는 것이 편하다.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [col for col in numeric_basic if col in X.columns]
categorical_features = [col for col in categorical_basic if col in X.columns]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="미상")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("numeric_features:", numeric_features)
print("categorical_features:", categorical_features)

numeric_features: ['OPERATING_YEARS', 'OPNNG_DAY_CO', 'LOG1P_LND_AR_VALUE', 'LOG1P_EDC_FCLTY_AR_VALUE', 'LOG1P_TOT_PROGRM_CO']
categorical_features: ['facility_type', 'CTPRVN_NM', 'MOBILE_PROVD_AT', 'SOUND_PROVD_AT']


## 11. 기준 모델 예시

아래 코드는 모델링 단계가 제대로 작동하는지 확인하기 위한 **기준 모델 예시**이다.  
최종 모델링은 별도 노트북 또는 다음 섹션에서 Ridge, RandomForest, GradientBoosting 등으로 확장한다.

In [12]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regression(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }

dummy_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

linear_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

ridge_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", Ridge(alpha=1.0))
])

results = {
    "Dummy_mean": evaluate_regression(dummy_model, X_train, X_test, y_train, y_test),
    "LinearRegression": evaluate_regression(linear_model, X_train, X_test, y_train, y_test),
    "Ridge": evaluate_regression(ridge_model, X_train, X_test, y_train, y_test),
}

result_df = pd.DataFrame(results).T.sort_values("RMSE")
display(result_df)

,MAE,RMSE,R2
Ridge,1.107783,1.374946,0.442973
LinearRegression,1.108871,1.376068,0.442063
Dummy_mean,1.482481,1.842294,-0.000052


## 12. 모델링용 데이터 저장 선택

아래 저장은 선택 사항이다.  
통계분석/머신러닝 노트북에서 같은 코드를 다시 실행할 예정이라면 저장하지 않아도 된다.

In [13]:
SAVE_OUTPUT = False

if SAVE_OUTPUT:
    output_dir = Path("data/processed")
    output_dir.mkdir(parents=True, exist_ok=True)

    # 목적변수 결측 제거 후 기본 모델용 데이터 저장
    basic_model_df = model_df[features_basic + [target, target_original]].copy()
    basic_model_df.to_csv(output_dir / "facility_modeling_basic.csv", index=False, encoding="utf-8-sig")

    extended_model_df = model_df[features_extended + [target, target_original]].copy()
    extended_model_df.to_csv(output_dir / "facility_modeling_extended.csv", index=False, encoding="utf-8-sig")

    staff_model_df = model_df[features_staff + [target, target_original]].copy()
    staff_model_df.to_csv(output_dir / "facility_modeling_staff.csv", index=False, encoding="utf-8-sig")

    print("저장 완료")
else:
    print("저장하지 않음")

저장하지 않음


## 13. 정리

이번 노트북에서 확정한 내용은 다음과 같다.

1. 목적 변수는 `LOG1P_VIEWNG_NMPR_CO`를 기본으로 사용한다.
2. `VIEWNG_NMPR_CO`는 원 단위 해석용으로만 사용한다.
3. 일평균 관람객 수 변수는 데이터 누수 위험이 있으므로 입력 변수에서 제외한다.
4. 목적변수 기반 이상치 플래그와 전체 이상치 카운트도 입력 변수에서 제외한다.
5. 기본 모델은 시설 유형, 지역, 운영연수, 개관일수, 시설 규모, 교육시설면적, 프로그램 수를 중심으로 구성한다.
6. 소장자료 수, 자료공간, 인원 수는 확장 모델에서 사용한다.
7. 인력 변수는 결측률이 높으므로 별도 실험 모델에서 사용한다.
8. 결측 처리는 train/test split 이후 파이프라인 안에서 수행한다.

다음 단계에서는 이 변수 세트를 바탕으로 다음 분석을 진행한다.

- 통계분석: 집단 간 차이 검정, 상관분석, 회귀분석
- 머신러닝: 기준 모델, 선형모델, 트리 기반 모델 비교

기본 변수 세트만으로도 로그 관람객 수에 대한 설명력이 일부 확인되었다.
다만 이는 기준 모델 수준의 결과이므로, 통계분석과 추가 머신러닝 모델 비교를 통해 변수 영향력과 예측 성능을 더 검토할 필요가 있다.

## 최종 정리

본 노트북에서는 EDA 결과를 바탕으로 통계분석 및 머신러닝 모델링 전에 필요한 변수 설계를 수행하였다.

목적변수는 관람객 수의 오른쪽 꼬리 분포를 고려하여 `LOG1P_VIEWNG_NMPR_CO`로 설정하였다.  
관람객 수 기반 파생 변수인 `DAY_AVRG_VIEWNG_NMPR_CO`, `LOG1P_DAY_AVRG_VIEWNG_NMPR_CO`와 관람객 수 기반 이상치 플래그는 데이터 누수 가능성이 있으므로 모델 입력 변수에서 제외하였다.

입력 변수는 기본 변수 세트, 자료·규모 확장 변수 세트, 인력 포함 실험 변수 세트로 구분하였다.  
기본 변수 세트에 대한 기준 모델 실행 결과, 평균 예측 모델보다 선형회귀와 Ridge 모델의 성능이 높게 나타나 선택한 변수들이 로그 관람객 수를 일부 설명할 가능성을 확인하였다.

다음 단계에서는 통계분석을 통해 시설 유형, 지역, 규모, 프로그램, 소장자료 수 등 주요 요인과 관람객 수 간의 관계를 검정하고, 이후 머신러닝 모델링을 통해 예측 성능과 변수 중요도를 비교한다.